<a href="https://colab.research.google.com/github/lyudmildimitrov1997-sudo/pppp/blob/main/Case%201_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import libraries

In [53]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

Random variable

In [54]:
random_variable = 42

Upload datasets

In [55]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
# Wir nutzen den direkten Pfad zu deinem Google Drive Hauptverzeichnis (MyDrive)
file_path = "/content/drive/MyDrive/"

try:
    data_a = pd.read_excel(file_path + "cybersecurity_assessment.xlsx")
    data_b = pd.read_excel(file_path + "supplier_master.xlsx")
    print("Erfolgreich geladen!")
except FileNotFoundError as e:
    print(f"Fehler beim Laden: {e}")
    print("\nHinweis: Falls die Dateien in einem Unterordner in deinem Google Drive liegen, passe den 'file_path' an (z. B. '/content/drive/MyDrive/MeinUnterordner/').")

Erfolgreich geladen!


Merge the two datasets

In [57]:
data = pd.merge(
    data_a,
    data_b,
    how="inner",
    on="supplier_id"
)

data.head()

,supplier_id,external_attack_surface_score,security_rating,average_patch_delay_days,mfa_coverage_pct,security_training_coverage_pct,prior_incidents_3y,fourth_party_dependency_count,iso27001_certified,incident_response_plan_tested,...,cybersecurity_incident_past_12m,supplier_name,industry,country,employees,annual_revenue_m_eur,years_as_supplier,business_criticality,access_level,sensitive_data_access
0,SUP-0001,66,54.0,93.0,58.0,69.0,0,5,False,False,...,True,Prime Digital 0001,Professional services,Germany,417,48.9,4,High,Portal only,True
1,SUP-0002,36,64.0,80.0,41.0,61.0,0,2,False,False,...,False,Evergreen Works 0002,Logistics,Poland,20,3.6,9,Medium,No access,False
2,SUP-0003,53,58.0,72.0,57.0,17.0,1,0,False,False,...,False,Blue Networks 0003,Industrial manufacturing,Mexico,276,43.2,8,Medium,Portal only,False
3,SUP-0004,17,67.0,50.0,76.0,75.0,0,1,False,False,...,False,Central Industries 0004,Electronics,Poland,103,22.0,2,Low,Portal only,False
4,SUP-0005,43,65.0,36.0,86.0,55.0,1,7,False,False,...,False,Global Supply 0005,IT services,Germany,809,68.4,5,Medium,Business systems,True


In [58]:
print("Dataset A:", data_a.shape)
print("Dataset B:", data_b.shape)
print("Merged dataset:", data.shape)

Dataset A: (1200, 13)
Dataset B: (1200, 10)
Merged dataset: (1200, 22)


Verify supplier IDs

In [59]:
print("Unique IDs in dataset A:",
      data_a["supplier_id"].nunique())

print("Unique IDs in dataset B:",
      data_b["supplier_id"].nunique())

print("Do all IDs match exactly?",
      set(data_a["supplier_id"]) == set(data_b["supplier_id"]))

print("Duplicate IDs in dataset A:",
      data_a["supplier_id"].duplicated().sum())

print("Duplicate IDs in dataset B:",
      data_b["supplier_id"].duplicated().sum())

Unique IDs in dataset A: 1200
Unique IDs in dataset B: 1200
Do all IDs match exactly? True
Duplicate IDs in dataset A: 0
Duplicate IDs in dataset B: 0


Class distibition

In [60]:
print("Percentage of suppliers with cybersecurity incidents:")

print(
    data["cybersecurity_incident_past_12m"]
    .value_counts(normalize=True) * 100
)

Percentage of suppliers with cybersecurity incidents:
cybersecurity_incident_past_12m
False    79.666667
True     20.333333
Name: proportion, dtype: float64


Explore missing values

In [61]:
print("Total rows with missing values:",
      data.isnull().any(axis=1).sum())

print("\nMissing values per column:")

print(data.isnull().sum()[data.isnull().sum() > 0])

Total rows with missing values: 176

Missing values per column:
security_rating                   30
average_patch_delay_days          41
mfa_coverage_pct                  46
security_training_coverage_pct    31
security_questionnaire_score      42
dtype: int64


In [62]:
data.dropna(axis=0).shape

(1024, 22)

In [63]:
data_cleaned = data.dropna(axis=0)

print("Original dataset:", data.shape)

print("Cleaned dataset:", data_cleaned.shape)

Original dataset: (1200, 22)
Cleaned dataset: (1024, 22)


Features and target variable

In [64]:
feature_cols = [
    "external_attack_surface_score",
    "security_rating",
    "average_patch_delay_days",
    "mfa_coverage_pct",
    "security_training_coverage_pct",
    "prior_incidents_3y",
    "fourth_party_dependency_count",
    "security_questionnaire_score"
]

X = data_cleaned[feature_cols]

y = data_cleaned["cybersecurity_incident_past_12m"].astype(int)

In [65]:
X.head()

,external_attack_surface_score,security_rating,average_patch_delay_days,mfa_coverage_pct,security_training_coverage_pct,prior_incidents_3y,fourth_party_dependency_count,security_questionnaire_score
0,66,54.0,93.0,58.0,69.0,0,5,45.0
1,36,64.0,80.0,41.0,61.0,0,2,51.0
2,53,58.0,72.0,57.0,17.0,1,0,58.0
3,17,67.0,50.0,76.0,75.0,0,1,86.0
4,43,65.0,36.0,86.0,55.0,1,7,82.0


In [66]:
y.value_counts()

,count
cybersecurity_incident_past_12m,
0,812
1,212


Train-test split

In [67]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=random_variable,
    stratify=y
)

In [68]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (819, 8)
X_test: (205, 8)
y_train: (819,)
y_test: (205,)


MACHINE LEARNING MODELS

Logistic Regression

In [69]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [70]:
log_reg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=random_variable
)

log_reg.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)

In [71]:
pred_lr = log_reg.predict(X_test_scaled)

prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

Decision Tree

In [72]:
clf_tree = DecisionTreeClassifier(
    random_state=random_variable,
    max_depth=5,
    min_samples_leaf=10,
    class_weight="balanced"
)

clf_tree.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', max_depth=5,
                       min_samples_leaf=10, random_state=42)

In [73]:
pred_dt = clf_tree.predict(X_test)

prob_dt = clf_tree.predict_proba(X_test)[:, 1]

Random Forest

In [74]:
rf = RandomForestClassifier(
    random_state=random_variable,
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=10,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=5,
                       min_samples_leaf=10, n_estimators=200, random_state=42)

In [75]:
pred_rf = rf.predict(X_test)

prob_rf = rf.predict_proba(X_test)[:, 1]

Compare models

In [76]:
model_comparison = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],

    "Accuracy": [
        accuracy_score(y_test, pred_lr),
        accuracy_score(y_test, pred_dt),
        accuracy_score(y_test, pred_rf)
    ],

    "Precision": [
        precision_score(y_test, pred_lr, zero_division=0),
        precision_score(y_test, pred_dt, zero_division=0),
        precision_score(y_test, pred_rf, zero_division=0)
    ],

    "Recall": [
        recall_score(y_test, pred_lr, zero_division=0),
        recall_score(y_test, pred_dt, zero_division=0),
        recall_score(y_test, pred_rf, zero_division=0)
    ]})

model_comparison = model_comparison.round(3)



In [77]:
print("\n" + "="*40)
print("LOGISTIC REGRESSION PERFORMANCE:")
print("Accuracy: ", accuracy_score(y_test, pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_lr))

print("\n" + "="*40)
print("DECISION TREE PERFORMANCE:")
print("Accuracy: ", accuracy_score(y_test, pred_dt))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_dt).T)

print("\n" + "="*40)
print("RANDOM FOREST PERFORMANCE:")
print("Accuracy: ", accuracy_score(y_test, pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_rf).T)


LOGISTIC REGRESSION PERFORMANCE:
Accuracy:  0.7268292682926829
Confusion Matrix:
 [[120  43]
 [ 13  29]]

DECISION TREE PERFORMANCE:
Accuracy:  0.624390243902439
Confusion Matrix:
 [[99 13]
 [64 29]]

RANDOM FOREST PERFORMANCE:
Accuracy:  0.7365853658536585
Confusion Matrix:
 [[128  19]
 [ 35  23]]


Model evaluation results

In [78]:
results = []

models = {
    "Logistic Regression": log_reg,
    "Decision Tree": clf_tree,
    "Random Forest": rf
}

for name, model in models.items():
    if name == "Logistic Regression":
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

Probability of the positive class

In [79]:
# Diese Berechnung wird nun vollständig in der Zelle darüber durchgeführt.
print("Die Modellauswertung wurde vorbereitet. Führe die nächste Zelle aus, um die Tabelle anzuzeigen!")

Die Modellauswertung wurde vorbereitet. Führe die nächste Zelle aus, um die Tabelle anzuzeigen!


Create comparison table

In [80]:
results_df = pd.DataFrame(results)

print("Model Performance Comparison:")
display(results_df.round(3))

Model Performance Comparison:


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.727,0.403,0.690,0.509,0.764
1,Decision Tree,0.624,0.312,0.690,0.430,0.661
2,Random Forest,0.737,0.397,0.548,0.460,0.735


Select the model with the highest F1-score

In [81]:
best_model_row = results_df.loc[results_df["F1-Score"].idxmax()]

print("\nRecommended Model:")
print("Model:", best_model_row["Model"])
print("F1-Score:", round(best_model_row["F1-Score"], 3))
print("Accuracy:", round(best_model_row["Accuracy"], 3))
print("Precision:", round(best_model_row["Precision"], 3))
print("Recall:", round(best_model_row["Recall"], 3))
print("ROC-AUC:", round(best_model_row["ROC-AUC"], 3))


Recommended Model:
Model: Logistic Regression
F1-Score: 0.509
Accuracy: 0.727
Precision: 0.403
Recall: 0.69
ROC-AUC: 0.764
